# Registration and metrics with module

In [ ]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from skimage.transform import resize, AffineTransform, warp

from tifffile import imread

from __future__ import print_function

import registration

#### Load data

In [ ]:
def load_image_data(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)[:, :]
                img_data.append(img)

    return img_data

def display_image(img):
    img = img.astype(float)
    img = img / img.max()
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.colorbar()                 
    plt.show()

In [ ]:
scale = 0.5023/0.209877

# load dapi
dapi_img_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/dapi")
dapi1_init = resize(dapi_img_data[0], (int(dapi_img_data[0].shape[0]/scale), int(dapi_img_data[0].shape[1]/scale)), anti_aliasing=True)
dapi2_init = resize(dapi_img_data[1], (int(dapi_img_data[1].shape[0]/scale), int(dapi_img_data[1].shape[1]/scale)), anti_aliasing=True)

# load hne
hne_image_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/hne")
hne1_init = hne_image_data[0]
hne2_init = hne_image_data[1]

display_image(dapi1_init)
display_image(hne1_init)

#### Preprocess images

In [ ]:
hne1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne1_init)
hne2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne2_init)
display_image(hne1_deconv)
display_image(hne2_deconv)

#### Register images

In [ ]:
import sys
sys.path.insert(0, r"D:\...\src\registration1")


In [ ]:
import registration
import importlib

importlib.reload(registration)
importlib.reload(registration.reg)

In [ ]:
def overlay_images(dapi_img, hne_img, title='overlay'):
    h, w = dapi_img.shape
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis([0, w, h, 0])
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)

def display_img_comparison(dapi_img, hne_img, hne_deconv, registered_hne_imgs, init_tform='initial similarity'):
    no_comparisons = len(registered_hne_imgs)
    h, w = dapi_img.shape
    
    plt.figure(figsize=((no_comparisons+3)*3, 6))

    plt.subplot(1, no_comparisons + 3, 1)
    plt.title('DAPI image')
    plt.imshow(dapi_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 2)
    plt.title('HnE image')
    plt.imshow(hne_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 3)
    plt.title('Initial Overlay')
    overlay_images(dapi_img, hne_deconv, title='initial overlay')

    tform_scheme = [init_tform, 'advanced rigid', 'advanced affine', 'advanced bspline']
    for i, tform in zip(range(no_comparisons), tform_scheme):
        plt.subplot(1, no_comparisons + 3, i + 4)
        overlay_images(dapi_img, registered_hne_imgs[i], title=tform)

    plt.tight_layout()
    plt.show()

In [ ]:
img1_tforms, img1_registered = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, 'similarity', 'bspline', 0.5023)

In [ ]:
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)

In [ ]:
img2_tforms, img2_registered = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, 'similarity', 'rigid', 0.5023)
display_img_comparison(dapi2_init, hne2_init, hne2_deconv,img2_registered)

In [ ]:
hne1_applying_tform = AffineTransform(scale=(1.0, 1.0), rotation=0.5, translation=(100, -200))
hne1_testing_img = warp(hne1_deconv, hne1_applying_tform)
display_image(hne1_testing_img)
img1_test_tforms, img1_test_registered = registration.register_DAPI_HnE(dapi1_init, hne1_testing_img, 'similarity', 'rigid', 0.5023)
display_img_comparison(dapi1_init, hne1_testing_img, hne1_deconv,img1_test_registered)